# Threat Modeling MCP and Agent Protocol Systems

Build and challenge a versioned threat model for the tenant support platform. This notebook is offline and credential-free. It evaluates the model artifact; it does not claim that planned controls are deployed or effective.

**Completion evidence:** eight components, eight directional flows, all six STRIDE prompts, a cross-boundary supply-chain attack path, model-lint failures, explicit metric populations, and a clear separation between design traceability and verified controls.

## 1. Use the four-question loop

1. What are we working on? Model components, assets, principals, zones, flows, assumptions, and dependencies.
2. What can go wrong? Use STRIDE, attack trees, abuse cases, agent/MCP catalogs, and system-specific attacker capabilities.
3. What will we do about it? Assign enforceable invariants, controls, owners, treatment, evidence, and review triggers.
4. Did we do a good enough job? Lint the model, execute tests, validate signals, exercise response, and review residual risk.

A taxonomy is a discovery aid. A diagram is a system hypothesis. Neither is runtime authorization or proof of safety.

In [ ]:
import runpy
from collections import Counter
from dataclasses import asdict, replace
from pathlib import Path

module = runpy.run_path(Path("lab.py"))
model = module["build_model"]()
summary = module["model_summary"](model)
summary

## 2. Inspect the system before listing threats

Every component records an operator, trust zone, and permitted authority. Every directional flow records data, protocol, boundary, identity source, authorization point, and audit event. A server request and a server result are separate flows because their security semantics differ.

In [ ]:
for component in model.components:
    print(f"{component.identifier:10} {component.trust_zone:18} {component.name}")

print("\nDirectional flows")
for flow in model.flows:
    print(f"{flow.identifier}: {flow.source} -> {flow.target} | {flow.trust_boundary} | authz={flow.authorization_point}")

assert len(model.components) == 8
assert len(model.flows) == 8
assert module["lint_model"](model) == ()

## 3. Make assumptions reviewable

An assumption is a dependency of the analysis. If an identity source, registry, deployment topology, credential scope, or downstream policy changes, the model must be reviewed rather than silently reused.

In [ ]:
for index, assumption in enumerate(model.assumptions, start=1):
    print(f"A-{index}: {assumption}")

assert any("authenticated host session" in item for item in model.assumptions)
assert any("untrusted data" in item for item in model.assumptions)

## 4. Apply STRIDE, then add domain-specific abuse paths

STRIDE helps reviewers ask six categories of questions. It does not rank risk or prove completeness. Each threat below is tied to a real flow, asset, attacker goal, prerequisites, multi-step abuse path, invariant, owner, and review trigger.

In [ ]:
for threat in model.threats:
    residual = module["risk_band"](threat.residual_likelihood, threat.impact)
    print(f"{threat.identifier} | {threat.stride:23} | {threat.flow_id} | residual={residual}")
    print("  path:", " -> ".join(threat.abuse_path))
    print("  invariant:", threat.invariant)

assert {threat.stride for threat in model.threats} == module["STRIDE"]

## 5. Find multi-boundary attack paths

A compromised supply-chain record can influence host selection, reach the server boundary, and eventually exercise downstream ticket authority. Graph traversal makes the path explicit; it does not establish exploitability by itself.

In [ ]:
paths = module["find_attack_paths"](model, "C-REGISTRY", "C-TICKET")
names = {component.identifier: component.name for component in model.components}
for path in paths:
    print(" -> ".join(names[node] for node in path))

expected = ("C-REGISTRY", "C-HOST", "C-SERVER", "C-TICKET")
assert expected in paths

## 6. Interpret metrics honestly

The numerator and denominator matter:

- **Design traceability:** threats with invariant, control, owner, trigger, and test/telemetry/runbook requirements ÷ all modeled threats.
- **Verified control coverage:** threats with a passed test, verified signal, and exercised runbook ÷ all modeled threats.
- **Open high residual threats:** high/critical post-control ratings that still require treatment.

The baseline should be structurally complete and operationally unverified.

In [ ]:
metrics = module["coverage_metrics"](model)
print(metrics)
assert metrics["design_traceability_percent"] == 100
assert metrics["verified_control_percent"] == 0
assert metrics["open_high_residual_threats"] == 3

## 7. Inject failures into the model

First remove a trust-boundary label. Then remove an evidence record. Structural lint and coverage should fail visibly instead of shrinking the denominator or reporting success.

In [ ]:
flow_without_boundary = replace(model.flows[0], trust_boundary="")
bad_boundary_model = replace(model, flows=(flow_without_boundary, *model.flows[1:]))
print(module["lint_model"](bad_boundary_model))

missing_evidence_model = replace(model, evidence=model.evidence[1:])
print(module["lint_model"](missing_evidence_model))
print(module["coverage_metrics"](missing_evidence_model))

assert "F-01: cross-zone flow has no trust boundary" in module["lint_model"](bad_boundary_model)
assert module["coverage_metrics"](missing_evidence_model)["design_complete_threats"] == 5

## 8. Failed evidence stays failed

Evidence history is immutable: recording a result returns a new model. The offline gate requires a receipt bound to the model version, evidence ID, configured producer, typed artifact URI, digest, and observation time. Production must also authenticate the producer and verify the artifact. A recorded failure must never count as verified control success.

In [ ]:
receipt = module["EvidenceReceipt"](
    identifier="RCPT-WORKSHOP-CROSS-TENANT",
    evidence_id="EV-cross-tenant-read-TEST",
    model_version=model.version,
    producer=module["TRUSTED_PRODUCERS"]["test"],
    status="failed",
    artifact_locator="ci://workshop/failure-injection/cross-tenant-read",
    artifact_digest="b" * 64,
    observed_at="2026-09-20T12:00:00Z",
)
failed = module["record_evidence"](model, receipt)
record = next(item for item in failed.evidence if item.identifier == "EV-cross-tenant-read-TEST")
print(asdict(record))
print(module["coverage_metrics"](failed))

assert record.status == "failed"
assert module["coverage_metrics"](failed)["verified_control_threats"] == 0
assert next(item for item in model.evidence if item.identifier == record.identifier).status == "not_run"

## 9. Risk bands are triage—not probability

The lab uses a documented 1–5 ordinal likelihood × impact matrix. Reviewers must retain the evidence and rationale for each input. A control written in the model does not justify lowering likelihood; control implementation and evidence do.

In [ ]:
risk_counts = Counter(
    module["risk_band"](threat.residual_likelihood, threat.impact)
    for threat in model.threats
)
print(risk_counts)
for threat in model.threats:
    print(threat.identifier, threat.risk_owner, threat.review_trigger)

assert risk_counts["high"] == 3
assert risk_counts["medium"] == 3

## 10. Professional tool choices

- **OWASP Threat Dragon 2.x:** collaborative graphical DFDs and versionable threat records.
- **OWASP pytm:** Python model-as-code with generated DFD, sequence, and report outputs.
- **Microsoft Threat Modeling Tool:** STRIDE-per-element design analysis.
- **Threagile:** YAML model-as-code and CI-oriented reporting.
- **Mermaid/diagrams.net:** communication only; add a separate traceability and evidence workflow.
- **OWASP, NIST, MITRE, MCP taxonomies and benchmarks:** threat-discovery and test inputs, not automatic production risk decisions.

Choose based on update workflow, stable IDs, code review, ownership, evidence links, export fidelity, and team adoption—not diagram aesthetics alone.

## 11. Production transfer and exercises

1. Add `ticket.update`; model approval binding, idempotency, uncertain outcomes, effect verification, rollback, and replay.
2. Add a local stdio server; include package, environment, filesystem, process, and egress boundaries.
3. Add a second MCP server plus A2A delegate; find identity/scope/provenance loss across the composed path.
4. Replace one planned evidence triplet with real CI, telemetry-validation, and exercise locators. Explain the bounded claim each artifact supports.
5. Move this system to Threat Dragon or pytm and report any stable IDs, fields, or evidence links lost in round-trip conversion.

**Reflection:** Which component can make the final ownership decision for `acme-7`, and why can neither the model nor the MCP server description supply that authority?